# [2.2] - DQN & VPG (solutions)

> **ARENA [Streamlit Page](https://arena-chapter2-rl.streamlit.app/02_[2.2]_DQN_&_VPG)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part2_q_learning_and_policy_gradient/2.2_DQN_&_VPG_exercises.ipynb?t=20260303) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part2_q_learning_and_policy_gradient/2.2_DQN_&_VPG_solutions.ipynb?t=20260303)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-22.png" width="350">

# Introduction

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network". This was used in a landmark paper [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf).

You'll also implement Vanilla Policy Gradient (VPG), the first policy gradient algorithm upon which many modern RL algorithms are based (including PPO).

## Content & Learning Objectives


### 1️⃣ DQN

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network". This was used in a landmark paper Playing Atari with [Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf).

You'll apply the technique of DQN to master the famous CartPole environment (below), and then (if you have time) move on to harder challenges like Acrobot and MountainCar.

> ##### Learning Objectives
>
> - Understand the DQN algorithm
> - Learn more about RL debugging, and build probe environments to debug your agents
> - Create a replay buffer to store environment transitions
> - Implement DQN using PyTorch, on the CartPole environment

### 2️⃣ VPG

The Policy Gradient Theorem is what all policy gradient methods are based on: it allows us to compute the gradient of the return, something that would naively not have a well defined gradient. We'll then implement Vanilla Policy Gradient (VPG) on the CartPole environment.

> ##### Learning Objectives
>
> - Understand the Policy Gradient Theorem
> - Understand the VPG algorithm: how to perform on-policy policy gradient
> - Implement VPG using PyTorch, on the CartPole environment

## Setup (don't read, just run!)

In [7]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter2_rl"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import jaxtyping
except:
    %pip wandb==0.18.7 einops "gymnasium[atari, accept-rom-license, other]==0.29.0" pygame jaxtyping

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/workspace/ARENA-3.0"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [8]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import sys
import time
import warnings
from collections import namedtuple
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import gymnasium as gym
import numpy as np
import torch as t
import torch.nn.functional as F
import wandb
from eindex import eindex
from gpu_env import CartPole
from gymnasium.spaces import Box, Discrete
from jaxtyping import Bool, Float, Int
from torch import Tensor, nn
from torchinfo import summary
from tqdm import tqdm

warnings.filterwarnings("ignore")

Arr = np.ndarray
ActType = Int
ObsType = Int

# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part2_q_learning_and_policy_gradient"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part2_q_learning_and_policy_gradient.tests as tests
import part2_q_learning_and_policy_gradient.utils as utils
from part1_intro_to_rl.utils import set_global_seeds
from part2_q_learning_and_policy_gradient.probe import Probe4, Probe5
from part2_q_learning_and_policy_gradient.utils import make_env
from plotly_utils import line, plot_cartpole_obs_and_dones
from rl_utils import generate_and_plot_trajectory, make_env

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

MAIN = __name__ == "__main__"

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


# 1️⃣ Deep Q-Learning

> ##### Learning Objectives
>
> - Understand the DQN algorithm
> - Learn more about RL debugging, and build probe environments to debug your agents
> - Create a replay buffer to store environment transitions
> - Implement DQN using PyTorch, on the CartPole environment

In this section, you'll implement Deep Q-Learning, often referred to as DQN for "Deep Q-Network". This was used in a landmark paper [Playing Atari with Deep Reinforcement Learning](https://www.cs.toronto.edu/~vmnih/docs/dqn.pdf).

At the time, the paper was very exciting: The agent would play the game by only looking at the same screen pixel data that a human player would be looking at, rather than a description of where the enemies in the game world are. The idea that convolutional neural networks could look at Atari game pixels and "see" gameplay-relevant features like a Space Invader was new and noteworthy. In 2022, we take for granted that convnets work, so we're going to focus on the RL aspect solely, and not the vision component.

## Optional Readings

* [Deep Q Networks Explained](https://www.lesswrong.com/posts/kyvCNgx9oAwJCuevo/deep-q-networks-explained) (25 minutes)
    * A high-level distillation as to how DQN works.
    * Read sections 1-4 (further sections optional).
* [Andy Jones - Debugging RL, Without the Agonizing Pain](https://andyljones.com/posts/rl-debugging.html) (10 minutes)
    * Useful tips for debugging your code when it's not working.
    * Read up to (not including) the Common Fixes section. Also read the Practical Advice section, up to and including "Use probe agents". The rest of the post is optional, and you're recommended to come back to it near the end if you're stuck.
    * The "probe environments" (a collection of simple environments of increasing complexity) section will be our first line of defense against bugs, you'll implement these in exercises below.

### Interesting Resources (not required reading)

- [An Outsider's Tour of Reinforcement Learning](http://www.argmin.net/2018/06/25/outsider-rl/) - comparison of RL techniques with the engineering discipline of control theory.
- [Towards Characterizing Divergence in Deep Q-Learning](https://arxiv.org/pdf/1903.08894.pdf) - analysis of what causes learning to diverge
- [Divergence in Deep Q-Learning: Tips and Tricks](https://amanhussain.com/post/divergence-deep-q-learning/) - includes some plots of average returns for comparison
- [Deep RL Bootcamp](https://sites.google.com/view/deep-rl-bootcamp/lectures) - 2017 bootcamp with video and slides. Good if you like videos.
- [DQN debugging using OpenAI gym Cartpole](https://adgefficiency.com/dqn-debugging/) - random dude's adventures in trying to get it to work.
- [CleanRL DQN](https://github.com/vwxyzjn/cleanrl) - single file implementations of RL algorithms. Your starter code today is based on this; try not to spoiler yourself by looking at the solutions too early!
- [Deep Reinforcement Learning Doesn't Work Yet](https://www.alexirpan.com/2018/02/14/rl-hard.html) - 2018 article describing difficulties preventing industrial adoption of RL.
- [Deep Reinforcement Learning Works - Now What?](https://tesslerc.github.io/posts/drl_works_now_what/) - 2020 response to the previous article highlighting recent progress.
- [Seed RL](https://github.com/google-research/seed_rl) - example of distributed RL using Docker and GCP.

### Conceptual overview of DQN

DQN is the natural extension of Q-Learning into the domain of deep learning. The main difference is that, instead of a table to store all the Q-values for each state-action pair, we train a neural network to learn this function for us. The usual implementation (which we'll use here) is for the Q-network to take the state as input, and output a vector of optimalQ-values for each action, i.e. we're learning the function:

$$
s \to (Q^*(s, a_1), ..., Q^*(s, a_n))
$$

Below is an algorithm showing the conceptual overview of DQN. We cycle through the following process:

* Generate a batch of experiences using our current policy, by **epsilon-greedy sampling** (i.e. we mostly take the action with the highest Q-value, but occasionally take a random action to encourage exploration). Store these experiences in the **replay buffer**.
* Use these values to calculate a **TD (temporal difference) error**, and update our network.
    * To increase stability, we also have a **target network** we use for the "next step" part of the TD error. This is a lagged copy of the Q-network (i.e. we update our Q-network via gradient descent, and then every so often we copy the Q-network weights over to our target network).
* Repeat this until convergence.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/ppo-alg-conceptual-2.png" width="750">

### Fast Feedback Loops

We want to have faster feedback loops, and learning from Atari pixels doesn't achieve that. It might take 15 minutes per training run to get an agent to do well on Breakout, and that's if your implementation is relatively optimized. Even waiting 5 minutes to learn Pong from pixels is going to limit your ability to iterate, compared to using environments that are as simple as possible.

### CartPole

The classic environment "CartPole-v1" is simple to understand, yet hard enough for a RL agent to be interesting, by the end of the day your agent will be able to do this and more! (Click to watch!)


[![CartPole](https://img.youtube.com/vi/46wjA6dqxOM/0.jpg)](https://www.youtube.com/watch?v=46wjA6dqxOM "CartPole")

If you'd like to try the CartPole environment yourself, [click here to open the simulation](https://davidquarel.github.io/cartpole) in a new tab. 
* Use Left/Right arrow keys to move the cart, 
* R to reset, 
* Q to quit. 
* Use F/S to make the simulation faster/slower. 

Unlike the real CartPole environment, this simulation will not terminate the episode if the pole falls over.
We've also cheated here and added a hidden third no-op action, such that if no button is pressed, no force is applied to the cart.
This makes the simulation a bit easier for you as the human to play.
The real cartpole environment doesn't act like this: the agent must choose to push the cart either left or right on each timestep.

The description of the task is [here](https://gymnasium.farama.org/environments/classic_control/cart_pole/). Note that unlike the previous environments, the observation here is now continuous. You can see the source for CartPole [here](https://github.com/Farama-Foundation/Gymnasium/blob/v0.29.0/gymnasium/envs/classic_control/cartpole.py); don't worry about the implementation but do read the documentation to understand the format of the actions and observations.

The simple physics involved would be very easy for a model-based algorithm to fit, (this is a common assignment in control theory using [proportional-integral-derivative](https://en.wikipedia.org/wiki/PID_controller) (PID) controllers) but today we're doing it model-free: your agent has no idea that these observations represent positions or velocities, and it has no idea what the laws of physics are. The network has to learn in which direction to bump the cart in response to the current state of the world.

Each environment can have different versions registered to it. By consulting [the Gym source](https://github.com/Farama-Foundation/Gymnasium/blob/v0.29.0/gymnasium/envs/__init__.py) you can see that CartPole-v0 and CartPole-v1 are the same environment, except that v1 has longer episodes. Again, a minor change like this can affect what algorithms score well; an agent might consistently survive for 200 steps in an unstable fashion that means it would fall over if ran for 500 steps.

In [9]:
env = gym.make("CartPole-v1", render_mode="rgb_array")

print(env.action_space)  # 2 actions: left and right
print(env.observation_space)  # Box(4): each action can take a continuous range of values

Discrete(2)
Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)


<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">Discrete(2)
Box([-4.8000002e+00 -3.4028235e+38 -4.1887903e-01 -3.4028235e+38], [4.8000002e+00 3.4028235e+38 4.1887903e-01 3.4028235e+38], (4,), float32)</pre>

### Outline of the Exercises

The exercises are roughly split into 4 sections:

1. Implement the Q-network that maps a state to an estimated value for each action.
2. Implement a replay buffer to store experiences $e_t = (s_t, a_t, r_{t+1}, d_{t+1}, s_{t+1})$.
3. Implement the policy which chooses actions based on the Q-network, plus epsilon greedy randomness to encourage exploration.
4. Piece everything together into a training loop and train your agent.

## The Q-Network

The Q-Network takes in an observation $s$ and outputs a vector $[Q^*(s, a^1), \ldots Q^*(s,a^n)]$ representing an estimate of the optimal Q-value for the given state $s$, and each possible action $\mathcal{A} = \{a^1, \ldots, a^n\}$. This replaces our Q-value table used in Q-learning.

For best results, the architecture of the Q-network can be customized to each particular problem. For example, [the architecture of OpenAI Five](https://cdn.openai.com/research-covers/openai-five/network-architecture.pdf) used to play DOTA 2 is pretty complex and involves LSTMs.

For learning from pixels, a simple convolutional network and some fully connected layers does quite well. Where we have already processed features here, it's even easier: an MLP of this size should be plenty large for any environment today.

Implement the Q-network using a standard MLP, constructed of alternating Linear and ReLU layers.
The size of the input will match the dimensionality of the observation space, and the size of the output will match the number of actions to choose from (associating a reward to each.)
The dimensions of the hidden_sizes are provided.

Here is a diagram of what our particular Q-Network will look like for CartPole (you can open it in a new tab if it's hard to see clearly):

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/mermaid-diagram-2024-11-28-200952.svg" style="background-color: #fffbe0; padding: 10px" width="160"><br>

<!-- 
flowchart TD
    A["Input (num_obs,)"] -> B["Linear(num_obs, 120)"]
    B -> C[ReLU]
    C -> D["Linear(120, 84)"]
    D -> E[ReLU]
    E -> F["Linear(84, num_actions)"]
    F -> G["Output (num_actions,)"]
{
  "theme": "default",
  "themeVariables": {
    "fontSize": "22px"
  }
}
-->

<details>
<summary>Question - why do we not include a ReLU at the end?</summary>

If you end with a ReLU, then your network can only predict 0 or positive Q-values. This will cause problems as soon as you encounter an environment with negative rewards, or you try to do some scaling of the rewards.

</details>

<details>
<summary>Question - since CartPole-v1 gives +1 reward on every timestep, why do you think the network doesn't just learn the constant +1 function regardless of observation?</summary>

The network is learning Q-values (the sum of all future expected discounted rewards from this state/action pair), not rewards. Correspondingly, once the agent has learned a good policy, the Q-value associated with state action pair (pole is slightly left of vertical, move cart left) should be large, as we would expect a long episode (and correspondingly lots of reward) by taking actions to help to balance the pole. Pairs like (cart near right boundary, move cart right) cause the episode to terminate, and as such the network will learn low Q-values.

</details>

### Exercise - implement `QNetwork`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Note - in this implementation we can assume that `obs_shape` is a tuple of length 1 (in the case of CartPole this will be `(4,)`), so you can treat it as just an integer value above, e.g. your first linear layer should be from `obs_shape[0]` to `120`.

In [10]:
class QNetwork(nn.Module):
    """
    For consistency with your tests, please wrap your modules in a `nn.Sequential` called `layers`.
    """

    layers: nn.Sequential

    def __init__(self, obs_shape: tuple[int], num_actions: int, hidden_sizes: list[int] = [120, 84]):
        super().__init__()
        assert len(obs_shape) == 1, "Expecting a single vector of observations"
        in_features_list = [obs_shape[0]] + hidden_sizes
        out_features_list = hidden_sizes + [num_actions]
        layers = []
        for i, (in_features, out_features) in enumerate(zip(in_features_list, out_features_list)):
            layers.append(nn.Linear(in_features, out_features))
            if i < len(in_features_list) - 1:
                layers.append(nn.ReLU())
        self.layers = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)


net = QNetwork(obs_shape=(4,), num_actions=2)
n_params = sum((p.nelement() for p in net.parameters()))
assert isinstance(getattr(net, "layers", None), nn.Sequential)
print(net)
print(f"Total number of parameters: {n_params}")
print("You should manually verify network is Linear-ReLU-Linear-ReLU-Linear")
assert not isinstance(net.layers[-1], nn.ReLU)
assert n_params == 10934

QNetwork(
  (layers): Sequential(
    (0): Linear(in_features=4, out_features=120, bias=True)
    (1): ReLU()
    (2): Linear(in_features=120, out_features=84, bias=True)
    (3): ReLU()
    (4): Linear(in_features=84, out_features=2, bias=True)
  )
)
Total number of parameters: 10934
You should manually verify network is Linear-ReLU-Linear-ReLU-Linear


## Replay Buffer

The goal of DQN is to reduce the reinforcement learning problem to a supervised learning problem. In supervised learning, training examples should be drawn **identically and independantly distributed (i.i.d.)** from some distribution, and we hope to generalize to future examples from that distribution. Obviously perfect i.i.d. sampling isn't attainable, but we can approximate this by filling a buffer of past experiences and sampling from it. Note that for very complex problems we may need a very large buffer, because we want the policy to get a representative sample of all the diverse scenarios that can happen in the environment. [OpenAI Five](https://cdn.openai.com/dota-2.pdf) used batch sizes of over 2 million experiences for Dota 2! However we'll be working with the fairly simple CartPole environment, and so we can get away with a much smaller buffer.

In RL, the distribution of experiences $e_t = (s_t, a_t, r_{t+1}, s_{t+1})$ to train from depend on the policy $\pi$ followed, which depends on the current state of the Q-value network, so DQN is always chasing a moving target. This is why the training loss curve isn't going to have a nice steady decrease like in supervised learning. We will extend experiences to $e_t = (s_t, a_t, r_{t+1}, s_{t+1}, d_{t+1})$. Here, $d_{t+1}$ is a boolean indicating that $s_{t+1}$ is a terminal observation, and that no further interaction happened beyond $s_{t+1}$ in the episode from which it was generated. 

### Termination vs Truncation

Note that we take $d_{t+1}$ to be `terminated`, not `done = terminated | truncated`. The reason is as follows: our time limit was imposed for practical reasons to help with learning, but if the agent views the environment timing out as a form of failure that terminates its reward then it would have no reason to prefer the behaviour "stay perfectly level" to the behaviour "stay level for the first 499 timesteps then immediately fall over"! We want to encourage the agent to perform well all the time, not just perform well until the environment times out. See [this page](https://farama.org/Gymnasium-Terminated-Truncated-Step-API#theory) for more discussion.

### `ReplayBuffer` and `ReplayBufferSamples`

We've given you 2 classes below. The first, `ReplayBuffer`, holds data from past experiences and also contains methods for sampling that data (the samples are instances of `ReplayBufferSamples`).

You should read these implementations carefully, making sure you understand how they work. A few things to note:

- The `add` method adds multiple experiences at once: the tensors like `obs` have shape `(num_envs, *obs_shape)`. This is because we're using the `SyncVectorEnv` class which allows us to step through & generate experiences for multiple environments simultaneously. We'll see how this works in practice later.
- The `add` method will add these experiences to the end of the buffer, slicing the buffer if it's too long. Note that the slicing is done so that we remove the oldest experiences when the buffer is full.
- The `sample` method will return a `ReplayBufferSamples` object containing the experiences sampled from the buffer. These are sampled with replacement, and the data is converted to PyTorch tensors on the correct device.

In [11]:
@dataclass
class ReplayBufferSamples:
    """
    Samples from the replay buffer, converted to PyTorch for use in neural network training.

    Data is equivalent to (s_t, a_t, r_{t+1}, d_{t+1}, s_{t+1}). Note - here, d_{t+1} is actually **terminated** rather
    than **done** (i.e. it records the times when we went out of bounds, not when the environment timed out).
    """

    obs: Float[Tensor, " sample_size *obs_shape"]
    actions: Float[Tensor, " sample_size *action_shape"]
    rewards: Float[Tensor, " sample_size"]
    terminated: Bool[Tensor, " sample_size"]
    next_obs: Float[Tensor, " sample_size *obs_shape"]


class ReplayBuffer:
    """
    Contains buffer; has a method to sample from it to return a ReplayBufferSamples object.
    """

    rng: np.random.Generator
    obs: Float[Arr, " buffer_size *obs_shape"]
    actions: Float[Arr, " buffer_size *action_shape"]
    rewards: Float[Arr, " buffer_size"]
    terminated: Bool[Arr, " buffer_size"]
    next_obs: Float[Arr, " buffer_size *obs_shape"]

    def __init__(
        self,
        num_envs: int,
        obs_shape: tuple[int],
        action_shape: tuple[int],
        buffer_size: int,
        seed: int,
    ):
        self.num_envs = num_envs
        self.obs_shape = obs_shape
        self.action_shape = action_shape
        self.buffer_size = buffer_size
        self.rng = np.random.default_rng(seed)

        self.obs = np.empty((0, *self.obs_shape), dtype=np.float32)
        self.actions = np.empty((0, *self.action_shape), dtype=np.int32)
        self.rewards = np.empty(0, dtype=np.float32)
        self.terminated = np.empty(0, dtype=bool)
        self.next_obs = np.empty((0, *self.obs_shape), dtype=np.float32)

    def add(
        self,
        obs: Float[Arr, " num_envs *obs_shape"],
        actions: Int[Arr, " num_envs *action_shape"],
        rewards: Float[Arr, " num_envs"],
        terminated: Bool[Arr, " num_envs"],
        next_obs: Float[Arr, " num_envs *obs_shape"],
    ) -> None:
        """
        Add a batch of transitions to the replay buffer.
        """
        # Check shapes & datatypes
        for data, expected_shape in zip(
            [obs, actions, rewards, terminated, next_obs],
            [self.obs_shape, self.action_shape, (), (), self.obs_shape],
        ):
            assert isinstance(data, np.ndarray)
            assert data.shape == (self.num_envs, *expected_shape)

        # Add data to buffer, slicing off the old elements
        self.obs = np.concatenate((self.obs, obs))[-self.buffer_size :]
        self.actions = np.concatenate((self.actions, actions))[-self.buffer_size :]
        self.rewards = np.concatenate((self.rewards, rewards))[-self.buffer_size :]
        self.terminated = np.concatenate((self.terminated, terminated))[-self.buffer_size :]
        self.next_obs = np.concatenate((self.next_obs, next_obs))[-self.buffer_size :]

    def sample(self, sample_size: int, device: t.device) -> ReplayBufferSamples:
        """
        Sample a batch of transitions from the buffer, with replacement.
        """
        indices = self.rng.integers(0, self.buffer_size, sample_size)

        return ReplayBufferSamples(
            obs=t.tensor(self.obs[indices], dtype=t.float32, device=device),
            actions=t.tensor(self.actions[indices], device=device),
            rewards=t.tensor(self.rewards[indices], dtype=t.float32, device=device),
            terminated=t.tensor(self.terminated[indices], device=device),
            next_obs=t.tensor(self.next_obs[indices], dtype=t.float32, device=device),
        )

Next, you can run the following code to visualize your cart's position and angle, and see how these look in both the buffer and the buffer's random samples. Do the samples look correctly shuffled? Also, based on the [CartPole source code](https://github.com/Farama-Foundation/Gymnasium/blob/v0.29.0/gymnasium/envs/classic_control/cartpole.py), do the angles & positions at which the cart terminates make sense? (Note, the min/max values in the table are different to the termination ranges, the latter can be found *below* the table in the docstring.)

Note that the code below uses the `SyncVectorEnv` class, which is what lets us step through multiple environments at once. We create it by passing it a list of functions which can be called to create environments (see the `make_env` function in `utils.py` for exactly how this works). Note that in this case we're just passing it a single environment; tomorrow we'll actually make full use of `SyncVectorEnv` by giving it multiple environments.

<!-- Lastly, note how the code below only adds experiences to the buffer if we didn't just finish the episode. The reason for this is as follows: the first time we call `envs.step` after terminating, it'll actually reset the environment rather than taking a step in it (see source code [here](https://github.com/Farama-Foundation/Gymnasium/blob/v1.0.0/gymnasium/vector/sync_vector_env.py#L209)). In other words, `obs` would be an out-of-bounds terminated state and `next_obs` would be a reset state. We don't want to add this experience to the buffer, because we don't want to teach our model that it can start from an out-of-bounds state and move to a reset state! You can see this reflected in the plots below - the vertical grey lines indicate values $t$ where $d_{t+1} = 1$, and we can see that immediately after this line both $s_t$ and $s_{t+1}$ reset to a new episode, without there being any intermediate logged state where $s_t$ refers to the old episode and $s_{t+1}$ to the new one. 

buffer = ReplayBuffer(num_environments=1, obs_shape=(4,), action_shape=(), buffer_size=256, seed=0)
envs = gym.vector.SyncVectorEnv([make_env("CartPole-v1", 0, 0, "test")])
obs, infos = envs.reset()
dones = np.array([False])

for i in range(256):
    # Choose random action, and take a step in the environment
    actions = envs.action_space.sample()
    next_obs, rewards, terminated, truncated, infos = envs.step(actions)
    next_dones = terminated | truncated

    # Add experience to buffer, as long as we didn't just finish an episode (so obs & next_obs are from the same episode)
    buffer.add(obs[~dones], actions[~dones], rewards[~dones], terminated[~dones], next_obs[~dones])
    obs = next_obs
    dones = next_dones
-->

<!-- Lastly, note how there's actually a slight epsiode mismatch between $s_t$ and $s_{t+1}$ in the plots below. The vertical lines show the values of $t$ where $d_{t+1} = 1$, so we can see that at these points $s_t$ and $s_{t+1}$ both refer to the terminated episode (they're the pre-final and final observations respectively), but immediately after these points $s_t$ refers to the terminal observation while $s_{t+1}$ refers to the first observation of the new episode. This isn't really a problem though, because we don't care what action our agent learns to take when $s_t$ is a terminal state (i.e. out of bounds). If we were being careful then we'd want to filter these terms out of our buffer, but for our purposes this week it doesn't really matter if we keep them in. -->

Lastly, note how when we terminate environments we do something slightly different. If `envs.step` results in some environments terminating, it'll actually return `next_obs` as the observation for the next environment. In this case, we want to use this as our starting observation for the next step, but we need to make sure we record the correct terminal observation in our buffer - we do this by extracting it from the `infos` dict, which is where it gets stored. You can see this in the plots below: the vertical lines are the values $t$ where $d_{t+1}=1$ i.e. $s_{t+1}$ is terminal, and we can see that $s_t, s_{t+1}$ both refer to the terminated episode at this point and both refer to the new episode immediately after.

In [12]:
buffer = ReplayBuffer(num_envs=1, obs_shape=(4,), action_shape=(), buffer_size=256, seed=0)
envs = gym.vector.SyncVectorEnv([make_env("CartPole-v1", 0, 0, "test")])
obs, infos = envs.reset()

for i in range(256):
    # Choose random action, and take a step in the environment
    actions = envs.action_space.sample()
    next_obs, rewards, terminated, truncated, infos = envs.step(actions)

    # Get `real_next_obs` by finding all environments where we terminated & replacing `next_obs`
    # with the actual terminal states
    true_next_obs = next_obs.copy()
    for n in range(envs.num_envs):
        if (terminated | truncated)[n]:
            true_next_obs[n] = infos["final_observation"][n]

    # Add experience to buffer, as long as we didn't just finish an episode (so obs & next_obs are
    # from the same episode)
    buffer.add(obs, actions, rewards, terminated, true_next_obs)
    obs = next_obs

sample = buffer.sample(256, device="cpu")

plot_cartpole_obs_and_dones(
    buffer.obs,
    buffer.terminated,
    title="Current obs s<sub>t</sub><br>so when d<sub>t+1</sub> = 1, these are the states just before termination",
)

plot_cartpole_obs_and_dones(
    buffer.next_obs,
    buffer.terminated,
    title="Next obs s<sub>t+1</sub><br>so when d<sub>t+1</sub> = 1, these are the terminated states",
)

plot_cartpole_obs_and_dones(
    sample.obs,
    sample.terminated,
    title="Current obs s<sub>t</sub> (sampled)<br>this is what gets fed into our model for training",
)

## Exploration

DQN makes no attempt to explore intelligently. The exploration strategy is the same as
for Q-Learning: agents take a random action with probability epsilon, but now we gradually
decrease epsilon. The Q-network is also randomly initialized (rather than initialized with zeros),
so its predictions of what is the best action to take are also pretty random to start.

Some games like [Montezuma's Revenge](https://paperswithcode.com/task/montezumas-revenge) have sparse rewards that require more advanced exploration methods to obtain. The player is required to collect specific keys to unlock specific doors, but unlike humans, DQN has no prior knowledge about what a key or a door is, and it turns out that bumbling around randomly has too low of a probability of correctly matching a key to its door. Even if the agent does manage to do this, the long separation between finding the key and going to the door makes it hard to learn that picking the key up was important.

As a result, DQN scored an embarrassing 0% of average human performance on this game.

### Reward Shaping

One solution to sparse rewards is to use human knowledge to define auxillary reward functions that are more dense and made the problem easier (in exchange for leaking in side knowledge and making
the algorithm more specific to the problem at hand). What could possibly go wrong?

The canonical example is for a game called [CoastRunners](https://openai.com/blog/faulty-reward-functions/), where the goal was given to maximize the
score (hoping that the agent would learn to race around the map). Instead, it found it could
gain more score by driving in a loop picking up power-ups just as they respawn, crashing and
setting the boat alight in the process.

### Reward Hacking

For Montezuma's Revenge, the reward was shaped by giving a small reward for
picking up the key.
One time this was tried, the reward was given slightly too early and the agent learned it could go close to the key without quite picking it up, obtain the auxillary reward, and then back up and repeat.

[![Montezuma Reward Hacking](https://img.youtube.com/vi/_sFp1ffKIc8/0.jpg)](https://www.youtube.com/watch?v=_sFp1ffKIc8 "Montezuma Reward Hacking")

A collected list of examples of Reward Hacking can be found [here](https://docs.google.com/spreadsheets/d/e/2PACX-1vRPiprOaC3HsCf5Tuum8bRfzYUiKLRqJmbOoC-32JorNdfyTiRRsR7Ea5eWtvsWzuxo8bjOxCG84dAg/pubhtml).


### Advanced Exploration

It would be better if the agent didn't require these auxillary rewards to be hardcoded by humans,
but instead reply on other signals from the environment that a state might be worth exploring. One idea is that a state which is "surprising" or "novel" (according to the agent's current belief
of how the environment works) in some sense might be valuable. Designing an agent to be
innately curious presents a potential solution to exploration, as the agent will focus exploration
in areas it is unfamiliar with. In 2018, OpenAI released [Random Network Distillation](https://openai.com/blog/reinforcement-learning-with-prediction-based-rewards/) which made progress in formalizing this notion, by measuring the agent's ability to predict the output of a neural network
on visited states. States that are hard to predict are poorly explored, and thus highly rewarded.
In 2019, an excellent paper [First return, then explore](https://arxiv.org/pdf/2004.12919v6.pdf) found an even better approach. Such reward shaping can also be gamed, leading to the
noisy TV problem, where agents that seek novelty become entranced by a source of randomness in the
environment (like a analog TV out of tune displaying white noise), and ignore everything else
in the environment.

### Exercise - implement linear scheduler

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 5-10 minutes on this exercise.
> ```

For now, implement the basic linearly decreasing exploration schedule.

In [13]:
def linear_schedule(
    current_step: int,
    start_e: float,
    end_e: float,
    exploration_fraction: float,
    total_timesteps: int,
) -> float:
    """
    Return the appropriate epsilon for the current step.

    Epsilon should be start_e at step 0 and decrease linearly to end_e at step (exploration_fraction
    * total_timesteps). In other words, we are in "explore mode" with start_e >= epsilon >= end_e
    for the first `exploration_fraction` fraction of total timesteps, and then stay at end_e for the
    rest of the episode.
    """
    return start_e + (end_e - start_e) * min(current_step / (exploration_fraction * total_timesteps), 1)


epsilons = [
    linear_schedule(step, start_e=1.0, end_e=0.05, exploration_fraction=0.5, total_timesteps=500)
    for step in range(500)
]
line(
    epsilons,
    labels={"x": "steps", "y": "epsilon"},
    title="Probability of random action",
    height=400,
    width=600,
)

tests.test_linear_schedule(linear_schedule)

All tests in `test_linear_schedule` passed!


### Epsilon Greedy Policy

In DQN, the policy is implicitly defined by the Q-network: we take the action with the maximum predicted reward. This gives a bias towards optimism. By estimating the maximum of a set of values $v_1, \ldots, v_n$ using the maximum of some noisy estimates $\hat{v}_1, \ldots, \hat{v}_n$ with $\hat{v}_i \approx v$, we get unlucky and get very large positive noise on some samples, which the maximum then chooses. Hence, the agent will choose actions that the Q-network is overly optimistic about.

See Sutton and Barto, Section 6.7 if you'd like a more detailed explanation, or the original [Double Q-Learning](https://proceedings.neurips.cc/paper/2010/file/091d584fced301b442654dd8c23b3fc9-Paper.pdf) paper which notes this maximisation bias, and introduces a method to correct for it using two separate Q-value estimators, each used to update the other.

### Exercise - implement the epsilon greedy policy

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

We've given you the first line of code, to convert the numpy array `obs` into a tensor on the correct device.
Note that we don't decide to explore for each environment individually: we either explore for all environments, or for none of them.
This means we can just avoid doing a forward pass for the Q-network entirely if we're exploring.

Other tips:

- Although you can technically use `envs.action_space.sample()` to sample actions, it's better practice to work with the random number generator `rng` that we've provided. You can use `rng.random()` to generate random numbers in the range $[0,1)$, and `rng.integers(0, n, size)` for an array of shape `size` random integers in the range $0, 1, \ldots, n-1$.
- Don't forget to convert the result back to a `np.ndarray`, as this function expects.
- Use `envs.single_action_space.n` to get the number of actions.

In [ ]:
def epsilon_greedy_policy(
    envs: gym.vector.SyncVectorEnv,
    q_network: QNetwork,
    rng: np.random.Generator,
    obs: Float[Arr, " num_envs *obs_shape"],
    epsilon: float,
) -> Int[Arr, " num_envs *action_shape"]:
    """
    With probability epsilon, take a random action. Otherwise, take a greedy action according to the
    q_network.

    Inputs:
        envs:       The family of environments to run against
        q_network:  The QNetwork used to approximate the Q-value function
        obs:        The current observation for each environment
        epsilon:    The probability of taking a random action

    Returns:
        actions:    The sampled action for each environment.
    """
    # Convert `obs` into a tensor so we can feed it into our model
    obs = t.from_numpy(obs).to(device)

    num_actions = envs.single_action_space.n
    if rng.random() < epsilon:
        return rng.integers(0, num_actions, size=(envs.num_envs,))
    else:
        q_scores = q_network(obs)
        return q_scores.argmax(-1).detach().cpu().numpy()


tests.test_epsilon_greedy_policy(epsilon_greedy_policy)

<details>
<summary>Help - I'm confused about the action shape here.</summary>

In our case, the action shape is `envs.single_action_space.shape = ()` (i.e. trivial, because our action is just a single integer not a vector or tensor) and the number of possible actions is `envs.single_action_space.n = 2`. This means your return type should just be a vector of ints of length `num_envs`, with each element being uniformly sampled from `[0, 1]`.

</details>

## Probe Environments

Extremely simple probe environments are a great way to debug your algorithm. The first one is given below.

Let's try and break down how this environment works. We see that the function `step` always returns the same thing. The observation and reward are always the same, and `done` is always true (i.e. the episode always terminates after one action). We expect the agent to rapidly learn that the value of the constant observation `[0.0]` is `+1`. This is in some sense the simplest possible probe.

### A note on action spaces

The space we're using here is `gym.spaces.Box`. This means we're dealing with real-valued quantities, i.e. continuous not discrete. The first two arguments of `Box` are `low` and `high`, and these define a box in $\mathbb{R}^n$. For instance, if these arrays are `(0, 0)` and `(1, 1)` respectively, this defines the box $0 \leq x, y \leq 1$ in 2D space.

In [ ]:
class Probe1(gym.Env):
    """
    One action, observation of [0.0], one timestep long, +1 reward.

    We expect the agent to rapidly learn that the value of the constant [0.0] observation is +1.0.
    Note we're using a continuous observation space for consistency with CartPole.
    """

    action_space: Discrete
    observation_space: Box

    def __init__(self, render_mode: str = "rgb_array"):
        super().__init__()
        self.observation_space = Box(np.array([0]), np.array([0]))
        self.action_space = Discrete(1)
        self.reset()

    def step(self, action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        return np.array([0]), 1.0, True, True, {}

    def reset(self, seed: int | None = None, options=None) -> ObsType | tuple[ObsType, dict]:
        super().reset(seed=seed)
        return np.array([0.0]), {}


gym.envs.registration.register(id="Probe1-v0", entry_point=Probe1)
env = gym.make("Probe1-v0")
assert env.observation_space.shape == (1,)
assert env.action_space.shape == ()

### Exercise - read & understand other probe environments

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-20 minutes here.
> ```

For each of the probes below, read their implementation code, and understand how they correspond to their docstrings (and to the [descriptions](https://andyljones.com/posts/rl-debugging.html#:~:text=Use%20probe%20environments) given in Andy Jones' post).

It's very important to understand how these probes work, and why they're useful tools for debugging. When you're working on your own RL projects, you might have to write your own probes to suit your particular use cases.

In [ ]:
class Probe2(gym.Env):
    """
    One action, observation of [-1.0] or [+1.0], one timestep long, reward equals observation.

    We expect the agent to rapidly learn the value of each observation is equal to the observation.
    """

    action_space: Discrete
    observation_space: Box

    def __init__(self, render_mode: str = "rgb_array"):
        super().__init__()
        self.observation_space = Box(np.array([-1.0]), np.array([+1.0]))
        self.action_space = Discrete(1)
        self.reset()
        self.reward = None

    def step(self, action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        assert self.reward is not None
        return np.array([self.observation]), self.reward, True, True, {}

    def reset(self, seed: int | None = None, options=None) -> ObsType | tuple[ObsType, dict]:
        super().reset(seed=seed)
        self.reward = 1.0 if self.np_random.random() < 0.5 else -1.0
        self.observation = self.reward
        return np.array([self.reward]), {}


class Probe3(gym.Env):
    """
    One action, [0.0] then [1.0] observation, two timesteps, +1 reward at the end.

    We expect the agent to rapidly learn the discounted value of the initial observation.
    """

    action_space: Discrete
    observation_space: Box

    def __init__(self, render_mode: str = "rgb_array"):
        super().__init__()
        self.observation_space = Box(np.array([-0.0]), np.array([+1.0]))
        self.action_space = Discrete(1)
        self.reset()

    def step(self, action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        self.n += 1
        if self.n == 1:
            return np.array([1.0]), 0.0, False, False, {}
        elif self.n == 2:
            return np.array([0.0]), 1.0, True, True, {}
        raise ValueError(self.n)

    def reset(self, seed: int | None = None, options=None) -> ObsType | tuple[ObsType, dict]:
        super().reset(seed=seed)
        self.n = 0
        return np.array([0.0]), {}


class Probe4(gym.Env):
    """
    Two actions, [0.0] observation, one timestep, reward is -1.0 or +1.0 dependent on the action.

    We expect the agent to learn to choose the +1.0 action.
    """

    action_space: Discrete
    observation_space: Box

    def __init__(self, render_mode: str = "rgb_array"):
        self.observation_space = Box(np.array([-0.0]), np.array([+0.0]))
        self.action_space = Discrete(2)
        self.reset()

    def step(self, action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        reward = -1.0 if action == 0 else 1.0
        return np.array([0.0]), reward, True, True, {}

    def reset(self, seed: int | None = None, options=None) -> ObsType | tuple[ObsType, dict]:
        super().reset(seed=seed)
        return np.array([0.0]), {}


class Probe5(gym.Env):
    """
    Two actions, random 0/1 observation, one timestep, reward is 1 if action equals observation,
    otherwise -1.

    We expect the agent to learn to match its action to the observation.
    """

    action_space: Discrete
    observation_space: Box

    def __init__(self, render_mode: str = "rgb_array"):
        self.observation_space = Box(np.array([-1.0]), np.array([+1.0]))
        self.action_space = Discrete(2)
        self.reset()

    def step(self, action: ActType) -> tuple[ObsType, float, bool, bool, dict]:
        reward = 1.0 if action == self.obs else -1.0
        return np.array([self.obs]), reward, True, True, {}

    def reset(self, seed: int | None = None, options=None) -> ObsType | tuple[ObsType, dict]:
        super().reset(seed=seed)
        self.obs = 1.0 if self.np_random.random() < 0.5 else 0.0
        return np.array([self.obs], dtype=float), {}


gym.envs.registration.register(id="Probe2-v0", entry_point=Probe2)
gym.envs.registration.register(id="Probe3-v0", entry_point=Probe3)
gym.envs.registration.register(id="Probe4-v0", entry_point=Probe4)
gym.envs.registration.register(id="Probe5-v0", entry_point=Probe5)

A brief summary of these, along with recommendations of where to go to debug if one of them fails (note that these won't be true 100% of the time, but should hopefully give you some useful direction):

<details>
<summary>Summary of probes</summary>

1. **Tests basic learning ability**. If this fails, it means the agent has failed to learn to associate a constant observation with a constant reward. You should check your loss functions and optimizers in this case.
2. **Tests the agent's ability to differentiate between 2 different observations (and learn their respective values)**. If this fails, it means the agent has issues with handling multiple possible observations.
3. **Tests the agent's ability to handle time & reward delay**. If this fails, it means the agent has problems with multi-step scenarios of discounting future rewards. You should look at how your agent step function works.
4. **Tests the agent's ability to learn from actions leading to different rewards**. If this fails, it means the agent has failed to change its policy for different rewards, and you should look closer at how your agent is updating its policy based on the rewards it receives & the loss function.
5. **Tests the agent's ability to map observations to actions**. If this fails, you should look at the code which handles multiple timesteps, as well as the code that handles the agent's map from observations to actions.

</details>

## Main DQN Algorithm

We now combine all the elements we have designed thus far into the final DQN algorithm. Here, we assume the environment returns three parameters $(s_{new}, r, d)$, a new state $s_{new}$, a reward $r$ and a boolean $d$ indicating whether interaction has terminated yet.

Our Q-value function $Q(s,a)$ is now a network $Q(s,a ; \theta)$ parameterised by weights $\theta$. The key idea, as in Q-learning, is to ensure the Q-value function satisfies the optimal Bellman equation
$$
Q(s,a ; \theta)
= \mathbb{E}_{s',r \sim p(\cdot \mid s,a)} \left[r + \gamma \max_{a'} Q(s', a' ;\theta) \right]
$$
which means the expected TD error will be zero (where expectation here is taken over randomly sampled trajectories):
$$
\mathbb{E} \left[ r_{t+1} + \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t) \right] = 0
$$
Note that we also have to deal with the case where the episode terminates: i.e. $s_{t+1}$ is a terminal state, and $d_{t+1} = 1$. Since the Q-value of a terminal state is always zero, we can just rewrite the expected TD error expression as:
$$
\mathbb{E} \left[ r_{t+1} + (1 - d_{t+1}) \, \gamma \max_a Q(s_{t+1}, a) - Q(s_t, a_t) \right] = 0
$$
since this makes sure the term is zero whenever $d_{t+1} = 1$. We can now see again why we used `terminated` not `terminated | truncated` here - we don't want the agent to learn that its value is always zero just before the episode ends and so there's no point in continuing to perform well!

Since we have an expression which should be zero in expectation for our true Q-value function, and we want the model to learn from a variety of experiences at once, we can sample batches of experiences $B = \{s_{t_i}, a_{t_i}, r_{t_i+1}, d_{t_i+1}, s_{t_i+1}\}_{i=1}^{|B|}$ from the replay buffer, and train against the loss function which equals the **squared temporal difference error**:
$$
L(\theta) = \frac{1}{|B|} \sum_{i=1}^B \left( r_{t_i+1} + (1 - d_{t_i+1}) \gamma \max_a Q(s_{t_i+1}, a ; \theta_\text{target}) - Q(s_{t_i}, a_{t_i} ; \theta) \right)^2
$$
Here, $\theta_\text{target}$ is a previous copy of the parameters $\theta$, so we're updating our $s_t$ estimates to catch up with our $s_{t+1}$ estimates (just like in standard Q-learning from earlier!). Every so often, we then update the target $\theta_\text{target} \leftarrow \theta$ as the agent improves it's Q-values from experience.

Below is the full DQN algorithm from a paper, for reference. The notation isn't identical to ours (e.g. they use an if/else statement to handle the terminal state case), but the basic algorithm is the same.

<img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/refs/heads/main/img/ch2-dqn-algo.png" width="700">

### DQN Dataclass

Below is a dataclass for training your DQN. You can use the `arg_help` method to see a description of each argument (it will also highlight any arguments which have been changed from their default values).

The exact breakdown of training is as follows:

* The agent takes `total_timesteps` steps in the environment during the training loop.
* The first `buffer_size` of these steps are used to fill the replay buffer (we don't update gradients until the buffer is full).
* After this point, we perform an optimizer step every `steps_per_train` steps of our agent. We also copy the weights from our Q-network to our target network every `trains_per_target_update` steps of our Q-network.

This is shown in the diagram below (the actual numbers aren't representative of the values in our dataclass, they're just to make sure the diagram is understandable - obviously the scale is very different in our actual training).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/dqn-breakdown.png" width="1200">

For example, in the code below we decrease `total_timesteps`, and this also decreases total training steps (which is computed in the `__post_init__` method of our dataclass, as a function of `total_timesteps`).

In [ ]:
@dataclass
class DQNArgs:
    # Basic / global
    seed: int = 1
    env_id: str = "CartPole-v1"
    num_envs: int = 1

    # Wandb / logging
    use_wandb: bool = False
    wandb_project_name: str = "DQNCartPole"
    wandb_entity: str | None = None
    video_log_freq: int | None = 50
    steps_per_live_video: int | None = None

    # Duration of different phases / buffer memory settings
    total_timesteps: int = 500_000
    steps_per_train: int = 10
    trains_per_target_update: int = 100
    buffer_size: int = 10_000

    # Optimization hparams
    batch_size: int = 128
    learning_rate: float = 2.5e-4

    # RL-specific
    gamma: float = 0.99
    exploration_fraction: float = 0.2
    start_e: float = 1.0
    end_e: float = 0.1

    def __post_init__(self):
        assert self.total_timesteps - self.buffer_size >= self.steps_per_train
        self.total_training_steps = (self.total_timesteps - self.buffer_size) // self.steps_per_train
        self.video_save_path = section_dir / "videos"


args = DQNArgs(total_timesteps=400_000)  # changing total_timesteps will also change ???
utils.arg_help(args)

### Exercise - fill in the agent class

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 25-45 minutes on this exercise.
> ```

You should now fill in the methods for the `DQNAgent` class below. This is a class which is designed to handle taking steps in the environment (with an epsilon greedy policy), and updating the buffer.

1. `play_step` should be somewhat similar to the demo code you saw earlier, which sampled a batch of experiences to add to the buffer. It should:
    - Get actions (using `self.get_actions` rather than randomly sampling like we did in the demo code before)
    - Step our environment with these actions
    - Add the new experiences to the buffer
        - Some of these observations 
    - Set your new observation as `self.obs`, ready for the next step
2. `get_actions` should do the following:
    - Set `self.epsilon` according to the linear schedule function & the current global step counter
    - Sample actions according to the epsilon-greedy policy (i.e. using your `epsilon_greedy_policy` function), and return them

A small note on code practices here - the implementation below was designed to follow **separation of concerns** (SoC), a design principle used in software engineering. The `DQNAgent` class only responsible for interacting with the environment; it doesn't do anything like create the Q-network or buffer on initialization. This is further reflected in the fact that we don't pass in `args` to our DQN agent, but instead pass in all the relevant variables separately (if we were forced to pass in `args`, this would be a sign that the DQN agent class might be doing too much work!).

In [ ]:
class DQNAgent:
    """Base Agent class handling the interaction with the environment."""

    def __init__(
        self,
        envs: gym.vector.SyncVectorEnv,
        buffer: ReplayBuffer,
        q_network: QNetwork,
        start_e: float,
        end_e: float,
        exploration_fraction: float,
        total_timesteps: int,
        rng: np.random.Generator,
    ):
        self.envs = envs
        self.buffer = buffer
        self.q_network = q_network
        self.start_e = start_e
        self.end_e = end_e
        self.exploration_fraction = exploration_fraction
        self.total_timesteps = total_timesteps
        self.rng = rng

        self.step = 0  # Tracking number of steps taken (across all environments)
        self.obs, _ = self.envs.reset()  # Need a starting observation
        self.epsilon = start_e  # Starting value (will be updated in `get_actions`)

    def play_step(self) -> dict:
        """
        Carries out a single interaction step between agent & environment, and adds results to the
        replay buffer.

        Returns `infos` (list of dictionaries containing info we will log).
        """
        self.obs = np.array(self.obs, dtype=np.float32)
        actions = self.get_actions(self.obs)
        next_obs, rewards, terminated, truncated, infos = self.envs.step(actions)

        # Get `real_next_obs` by finding all environments where we terminated & replacing `next_obs`
        # with the actual terminal states
        true_next_obs = next_obs.copy()
        for n in range(self.envs.num_envs):
            if (terminated | truncated)[n]:
                true_next_obs[n] = infos["final_observation"][n]

        self.buffer.add(self.obs, actions, rewards, terminated, true_next_obs)
        self.obs = next_obs

        self.step += self.envs.num_envs
        return infos

    def get_actions(self, obs: np.ndarray) -> np.ndarray:
        """
        Samples actions according to the epsilon-greedy policy using the linear schedule for epsilon.
        """
        self.epsilon = linear_schedule(
            self.step, self.start_e, self.end_e, self.exploration_fraction, self.total_timesteps
        )
        actions = epsilon_greedy_policy(self.envs, self.q_network, self.rng, obs, self.epsilon)
        assert actions.shape == (len(self.envs.envs),)
        return actions


tests.test_agent(DQNAgent)

Before we move on to the big exercise of today (completing the `DQNTrainer` class), we'll briefly discuss logging to Weights and Biases in RL, plus some general advice on what kinds of variables you should be logging.

### Logging to `wandb` in RL

In previous exercises in this chapter, we've just trained the agent, and then plotted the reward per episode after training. For small toy examples that train in a few seconds this is fine, but for longer runs we'd like to watch the run live and make sure the agent is doing something interesting (especially if we were planning to run the model overnight). Luckily, **Weights and Biases** has got us covered! When you run your experiments, you'll be able to view not only *live plots* of the loss and average reward per episode while the agent is training - you can also log and view animations, which visualise your agent's progress in real time! The code below will handle all logging.

Sadly, effective logging & debugging in RL isn't just about watching videos, since in the vast majority of cases where your algorithm has a bug, the agent will just fail to learn anything useful and the videos won't be informative. Debugging RL requires knowing what variables to log and how to interpret the results you're getting, which requires some understanding of the underlying theory! This is part of the reason why we've spent so much time discussing the theory behind DQN and other RL algorithms, rather than just giving you a black box to train.

As an example of how logged variables can be misleading and hard to interpret, consider our TD loss function in DQN. This loss function just reflects how close together the Q-network's estimates are to the experiences currently sampled from the replay buffer, which might not adequately represent what the world actually looks like. This means that once the agent starts to learn something and do better at the problem, it's expected for the loss to increase. For example, maybe the Q-network initially learned some state was bad, because an agent that reached them was just flapping around randomly and died shortly after. But now it's getting evidence that the same state is good, now that the agent that reached the state has a better idea what to do next. A higher loss is thus actually a good sign that something is happening (the agent hasn't stagnated), but it's not clear if it's learning anything useful without also checking how the total reward per episode has changed. Key point - **just looking at one variable can be misleading, we need to log multiple variables and derive a picture of what's happening from taking all of them into account!**

Some useful variables to log during DQN training are:

- TD loss, i.e. the actual loss you're backpropagating through. This should start off high and decrease pretty quickly, but may not be monotonic (i.e. temporary spikes in loss aren't necessarily a bad thing)
- SPS (steps per second), i.e. the total number of agent steps divided by the total time. This helps us debug when the environment steps are a bottleneck (won't be the case in a simple environment like this one, but might matter more when we move to more complex environments)
- Q-values, i.e. the predicted Q-values from the Q-network. Can you guess how these should behave?

<details>
<summary>Question - what do you think the Q values will do when the agent moves closer to solving the cartpole environment?</summary>

Initially they should be near zero, thanks to the randomly initialized model weights. As our episode length get closer to 500 (i.e. we can essentially solve the environment), they should tend to the limit of the total possible time-discounted reward available, which is the geometric sum $1 + \gamma + \gamma^2 + \cdots$ (since we get 1 reward for every second we stand up, and as previously discussed, the way we handle `dones` in the formula above doesn't assume a truncated environment causes future rewards to be terminated). The limit of this sum is $\frac{1}{1-\gamma}$, which for our default value $\gamma = 0.99$ is approximately 100.

Note, the Q values won't increase smoothly, they'll spike up immediately after we copy over the weights from our Q-network to our target network. This is because each time we copy over weights, our gradient changes and the Q-network rapidly "catches up" to this new target network, causing the Q values to change rapidly. However, our copying over of weights will be frequent enough that these jumps will be relatively small, and so the curve should still appear smooth.

</details>

### Exercise - write DQN training loop

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 30-60 minutes on this exercise.
> ```

Now we'll create a new class `DQNTrainer`, which will handle the full training loop. We've filled in the `__init__` for you, which defines all the things you need (the networks, optimizer, replay buffer, and the agent). We've also filled in `train` for you, which performs the main training loop: it optionally initializes Weights & Biases, fills the buffer using `prepopulate_replay_buffer`, then alternates between training steps (where we sample from the buffer) & adding to the buffer (adding `args.train_frequncy`).

You should fill in the remaining 2 methods. First you should get the basic no-logging version working, then once you're running without error (even if maybe you're not learning anything useful) you should move onto logging as this will help you debug.

- `add_to_replay_buffer`
    - This calls `self.agent.play_step()` to take `n` steps in the environment, which adds the results to the replay buffer
    - It's used to fill the buffer before training starts, and before each training step to add new experiences to the buffer
- `training_step`
    - This performs an update step from a batch of experiences from the buffer, sampled using `self.buffer.sample` with batch size `self.args.batch_size`
    - An update step involves:
        - Getting the predicted Q-values $Q(s_{t_i}, a_{t_i} ; \theta)$ from the Q-network
        - Getting the max target Q-values $\max_a Q(s_{t_i+1}, a ; \theta_\text{target})$ from the target network (remember to use inference mode - we're not training the target network!)
        - Computing the TD loss $L(\theta)$ using the formula we gave earlier (we've also copied it below, for convenience)
        - Performing an update step with this loss
    - You should also copy weights from the Q-network to the target network every `args.trains_per_target_update` steps (i.e. whenever `self.agent.step` is a multiple of this). The `load_state_dict` method might be useful here

For convenience, here's the full TD loss formula again:
$$
L(\theta) = \frac{1}{|B|} \sum_{i=1}^B \left( r_{t_i+1} + (1 - d_{t_i+1}) \gamma \max_a Q(s_{t_i+1}, a ; \theta_\text{target}) - Q(s_{t_i}, a_{t_i} ; \theta) \right)^2
$$

When you get to logging, there are 2 types of data you can log:

- Data for terminated episodes, during buffer filling
    - Terminated episode data can be found in the `infos` dict returned by the `agent.play_step` method. If environment `env_idx` terminated, then `infos["final_info"][env_idx]["episode"]` will be a dict containing the length `l` and reward `r` of the terminated episode
        - We've given you a helper function `get_episode_data_from_infos` which gives you a dict of the episode length & reward for the first terminated env, or `None` if no envs terminated. See the [documentation page](https://gymnasium.farama.org/v0.29.0/_modules/gymnasium/wrappers/record_episode_statistics/) for an explanation.
    - You can also log the SPS (steps per second) if you like, this helps figure out if the environment transitions are the bottleneck for your algorithm
- Data during training steps
    - Mean TD loss, Q values, and the epsilon hyperparameter are all useful to log

Don't be discouraged if your code takes a while to work - it's normal for debugging RL to take longer than you would expect. Add asserts or your own tests, implement an appropriate probe environment, try anything in the Andy Jones post that sounds promising, and try to notice confusion. Reinforcement Learning is often so tricky as even if the algorithm has bugs, the agent might still learn something useful regardless (albeit maybe not as well), or even if everything is correct, the agent might just fail to learn anything useful (like how DQN failed to do anything on Montezuma's Revenge.)

Since the environment is already known to be one DQN can solve, and we've already provided hyperparameters that work for this environment, hopefully that's isolated a lot of the problems one would usually have with solving real world problems with RL.

In [ ]:
def get_episode_data_from_infos(infos: dict) -> dict[str, int | float] | None:
    """
    Helper function: returns dict of data from the first terminated environment, if at least one
    terminated.
    """
    for final_info in infos.get("final_info", []):
        if final_info is not None and "episode" in final_info:
            return {
                "episode_length": final_info["episode"]["l"].item(),
                "episode_reward": final_info["episode"]["r"].item(),
                "episode_duration": final_info["episode"]["t"].item(),
            }


class DQNTrainer:
    def __init__(self, args: DQNArgs):
        set_global_seeds(args.seed)
        self.args = args
        self.rng = np.random.default_rng(args.seed)
        self.run_name = f"{args.env_id}__{args.wandb_project_name}__seed{args.seed}__{time.strftime('%Y%m%d-%H%M%S')}"
        self.envs = gym.vector.SyncVectorEnv(
            [make_env(idx=idx, run_name=self.run_name, **args.__dict__) for idx in range(args.num_envs)]
        )

        # Define some basic variables from our environment (note, we assume a single discrete action space)
        num_envs = self.envs.num_envs
        action_shape = self.envs.single_action_space.shape
        num_actions = self.envs.single_action_space.n
        obs_shape = self.envs.single_observation_space.shape
        assert action_shape == ()

        # Create our replay buffer
        self.buffer = ReplayBuffer(num_envs, obs_shape, action_shape, args.buffer_size, args.seed)

        # Create our networks & optimizer (target network should be initialized with a copy of the Q-network's weights)
        self.q_network = QNetwork(obs_shape, num_actions).to(device)
        self.target_network = QNetwork(obs_shape, num_actions).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.optimizer = t.optim.AdamW(self.q_network.parameters(), lr=args.learning_rate)

        # Create our agent
        self.agent = DQNAgent(
            self.envs,
            self.buffer,
            self.q_network,
            args.start_e,
            args.end_e,
            args.exploration_fraction,
            args.total_timesteps,
            self.rng,
        )

    def add_to_replay_buffer(self, n: int, verbose: bool = False):
        """
        Takes n steps with the agent, adding to the replay buffer (and logging any results). Should
        return a dict of data from the last terminated episode, if any.

        Optional argument `verbose`: if True, we can use a progress bar (useful to check how long
        the initial buffer filling is taking).
        """
        data = None
        t0 = time.time()

        for step in tqdm(range(n), disable=not verbose, desc="Adding to replay buffer"):
            infos = self.agent.play_step()

            # Get data from environments, and log it if some environment did actually terminate
            new_data = get_episode_data_from_infos(infos)
            if new_data is not None:
                data = new_data  # makes sure we return a non-empty dict, if some episode terminates
                if self.args.use_wandb:
                    wandb.log(new_data, step=self.agent.step)

        # Log SPS
        if self.args.use_wandb:
            wandb.log({"SPS": (n * self.envs.num_envs) / (time.time() - t0)}, step=self.agent.step)

        return data

    def prepopulate_replay_buffer(self):
        """
        Called to fill the replay buffer before training starts.
        """
        n_steps_to_fill_buffer = self.args.buffer_size // self.args.num_envs
        self.add_to_replay_buffer(n_steps_to_fill_buffer, verbose=True)

    def training_step(self, step: int) -> None:
        """
        Samples once from the replay buffer, and takes a single training step.

        Args:
            step (int): The number of training steps taken (used for logging, and for deciding when
            to update the target network)
        """
        data = self.buffer.sample(self.args.batch_size, device)  # s_t, a_t, r_{t+1}, d_{t+1}, s_{t+1}

        with t.inference_mode():
            target_max = self.target_network(data.next_obs).max(-1).values
        predicted_q_vals = self.q_network(data.obs)[range(len(data.actions)), data.actions]

        td_error = data.rewards + self.args.gamma * target_max * (1 - data.terminated.float()) - predicted_q_vals
        loss = td_error.pow(2).mean()
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()

        if step % self.args.trains_per_target_update == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())

        if self.args.use_wandb:
            wandb.log(
                {
                    "td_loss": loss,
                    "q_values": predicted_q_vals.mean().item(),
                    "epsilon": self.agent.epsilon,
                },
                step=self.agent.step,
            )

    def train(self) -> None:
        if self.args.use_wandb:
            wandb.init(
                project=self.args.wandb_project_name,
                entity=self.args.wandb_entity,
                name=self.run_name,
                monitor_gym=self.args.video_log_freq is not None,
            )
            wandb.watch(self.q_network, log="all", log_freq=50)

        self.prepopulate_replay_buffer()

        pbar = tqdm(range(self.args.total_training_steps))
        last_logged_time = time.time()  # so we don't update the progress bar too much

        for step in pbar:
            data = self.add_to_replay_buffer(self.args.steps_per_train)
            if data is not None and time.time() - last_logged_time > 0.5:
                last_logged_time = time.time()
                pbar.set_postfix(**data)

            self.training_step(step)

            if self.args.steps_per_live_video is not None and step % self.args.steps_per_live_video == 0:
                from IPython.display import display

                html_animation = generate_and_plot_trajectory(self, self.args)
                display(html_animation)

        self.envs.close()
        if self.args.use_wandb:
            wandb.finish()

NameError: name 'DQNArgs' is not defined

Here's some boilerplate code to test out your various probes, which you should make sure you're passing before testing on Cartpole.

In [ ]:
def test_probe(probe_idx: int):
    """
    Tests a probe environment by training a network on it & verifying that the value functions are
    in the expected range.
    """
    # Train our network on this probe env
    args = DQNArgs(
        env_id=f"Probe{probe_idx}-v0",
        wandb_project_name=f"test-probe-{probe_idx}",
        total_timesteps=3000 if probe_idx <= 2 else 5000,
        learning_rate=0.001,
        buffer_size=500,
        use_wandb=False,
        trains_per_target_update=20,
        video_log_freq=None,
    )
    trainer = DQNTrainer(args)
    trainer.train()

    # Get the correct set of observations, and corresponding values we expect
    obs_for_probes = [[[0.0]], [[-1.0], [+1.0]], [[0.0], [1.0]], [[0.0]], [[0.0], [1.0]]]
    expected_value_for_probes = [
        [[1.0]],
        [[-1.0], [+1.0]],
        [[args.gamma], [1.0]],
        [[-1.0, 1.0]],
        [[1.0, -1.0], [-1.0, 1.0]],
    ]
    tolerances = [5e-4, 5e-4, 5e-4, 5e-4, 1e-3]
    obs = t.tensor(obs_for_probes[probe_idx - 1]).to(device)

    # Calculate the actual value, and verify it
    value = trainer.q_network(obs)
    expected_value = t.tensor(expected_value_for_probes[probe_idx - 1]).to(device)
    t.testing.assert_close(value, expected_value, atol=tolerances[probe_idx - 1], rtol=0)
    print("Probe tests passed!\n")


for probe_idx in range(1, 6):
    test_probe(probe_idx)

Once you've passed the tests for all 5 probe environments, you should test your model on Cartpole. We recommend you start by not using wandb until you can get it running without error, because this will improve your feedback loops (however if you've passed all probe environments then there's a good chance this code will just work for you).

In [ ]:
args = DQNArgs(use_wandb=True, steps_per_live_video=5_000)
trainer = DQNTrainer(args)
trainer.train()

### Catastrophic forgetting

Note - you might see performance frequently drop off after it's achieved the maximum for a while, before eventually recovering again and repeating the cycle. Here's an example CartPole run using the solution code:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/cf3.png" width="550">

This is a well-known RL phenomena called **catastrophic forgetting**. It happens when the replay buffer mostly contains successful experiences, and the model forgets how to adapt or recover from bad states. One way to fix this is to change your buffer to keep 10 of experiences from previous epochs, and 90% of experiences from the current phase. Can you implement this?

When we cover PPO tomorrow, we'll also introduce **reward shaping**, which is another way this kind of behaviour can be mitigated.

## Beyond CartPole

If things go well and your agent masters CartPole, the next harder challenges are [Acrobot-v1](https://github.com/Farama-Foundation/Gymnasium/blob/v0.29.0/gymnasium/envs/classic_control/acrobot.py), and [MountainCar-v0](https://github.com/Farama-Foundation/Gymnasium/blob/v0.29.0/gymnasium/envs/classic_control/mountain_car.py). These also have discrete action spaces, which are the only type we're dealing with today. Feel free to Google for appropriate hyperparameters for these other problems - in a real RL problem you would have to do hyperparameter search using the techniques we learned on a previous day because bad hyperparameters in RL often completely fail to learn, even if the algorithm is perfectly correct.

There are many more exciting environments to play in, but generally they're going to require more compute and more optimization than we have time for today. If you finish the main material, some we recommend are:

- [Minimalistic Gridworld Environments](https://github.com/Farama-Foundation/gym-minigrid) - a fast gridworld environment for experiments with sparse rewards and natural language instruction.
- [microRTS](https://github.com/santiontanon/microrts) - a small real-time strategy game suitable for experimentation.
- [Megastep](https://andyljones.com/megastep/) - RL environment that runs fully on the GPU (fast!)
- [Procgen](https://github.com/openai/procgen) - A family of 16 procedurally generated gym environments to measure the ability for an agent to generalize. Optimized to run quickly on the CPU.
- [Atari](https://ale.farama.org/environments/) - although you might want to wait until tomorrow to try this on DQN, because we'll be going through some guided exercises implementing Atari with PPO tomorrow!

<details>
<summary>Some (very unpolished) code for setting up Atari with DQN</summary>

This is based on a hybrid of tomorro's agent/critic network setup for Atari, and the DQN implementation in this notebook. I've achieved decent performance in 40 mins training this, but not as good as we get when we do PPO on Atari tomorrow, so I think this is somewhat underoptimized - if anyone finds improvements then feel free to make a PR!

```python
def layer_init(layer: nn.Linear, std=np.sqrt(2), bias_const=0.0):
    t.nn.init.orthogonal_(layer.weight, std)
    t.nn.init.constant_(layer.bias, bias_const)
    return layer

class QNetwork(nn.Module):
    layers: nn.Sequential

    def __init__(self, obs_shape: tuple[int, ...], num_actions: int, hidden_sizes: list[int] = [120, 84]):
        super().__init__()

        assert len(obs_shape) == 3, "We're only supporting Atari for now, obs should be RGB images"

        assert obs_shape[-1] % 8 == 4
        L_after_convolutions = (obs_shape[-1] // 8) - 3
        in_features = 64 * L_after_convolutions * L_after_convolutions

        self.layers = nn.Sequential(
            layer_init(nn.Conv2d(4, 32, 8, stride=4, padding=0)),
            nn.ReLU(),
            layer_init(nn.Conv2d(32, 64, 4, stride=2, padding=0)),
            nn.ReLU(),
            layer_init(nn.Conv2d(64, 64, 3, stride=1, padding=0)),
            nn.ReLU(),
            nn.Flatten(),
            layer_init(nn.Linear(in_features, 512)),
            nn.ReLU(),
            layer_init(nn.Linear(512, num_actions), std=0.01),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)

args = DQNArgs(
    use_wandb=True,
    buffer_size=1000,
    batch_size=32,
    end_e=0.01,
    learning_rate=1e-4,
    total_timesteps=20_000,
    steps_per_train=5,
    mode="atari",
    env_id="ALE/Breakout-v5",
    wandb_project_name="DQNAtari",
    num_envs=4,
)
trainer = DQNTrainer(args)
trainer.train()
```


</details>

## Bonus

### Target Network

Why have the target network? Modify the DQN code above, but this time use the same network for both the target and the Q-value network, rather than updating the target every so often.

Compare the performance of this against using the target network.

### Shrink the Brain

Can DQN still learn to solve CartPole with a Q-network with fewer parameters? Could we get away with three-quarters or even half as many parameters? Try comparing the resulting training curves with a shrunken version of the Q-network. What about the same number of parameters, but with more/less layers, and less/more parameters per layer?

### Dueling DQN

Implement dueling DQN according to [the paper](https://arxiv.org/pdf/1511.06581.pdf) and compare its performance.

# 2️⃣ VPG

> ##### Learning Objectives
>
> - Understand the Policy Gradient Theorem
> - Understand the VPG algorithm: how to perform on-policy policy gradient
> - Implement VPG using PyTorch, on the CartPole environment

## Policy Gradient Theorem

Instead of learning action-values and deriving a policy (as in Q-learning or DQN), **policy gradient methods learn the policy directly**.
- Policy is parameterized: $\pi_\theta(a|s)$ with parameters $\theta$ (often a neural network).  
- Objective: Choose $\theta$ to maximize expected return $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[G(\tau)]$ (joy), where $\tau$ is a trajectory and $G(\tau)$ its return.  

We would desire to update the policy directly via **gradient ascent** against $J(\theta)$:
$$
\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)
$$

The problem is that the return is a sum of rewards from the trajectory, and the trajectory itself is a result of sampling from the policy, over and over, 
as well as being dependant on the environmental distribution, which we do not have access to.
There is no clear way to directly compute the gradient of the return with respect to the policy parameters.
The solution here is the **policy gradient theorem**, which states that we can instead use the log-probability weighted return as an unbiased estimator of the gradient of the return.

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_t G_t \nabla_\theta \log \pi_\theta(a_t|s_t) \right]
$$

<details>
<summary>Derivation</summary>

The probability of sampling a trajectory 
$\tau = (s_0, a_0, s_1, a_1, \dots, s_T)$ 
is given by
$$
\Pr(\tau|\theta) = \prod_{t=0}^{T-1} \pi_\theta(a_t|s_t)\,\mu(s_{t+1}|s_t, a_t)
$$
where $\mu$ is the environment transition probability.

$$
\begin{align*}
   \nabla_\theta J(\theta) &= \nabla_\theta \mathbb{E}_{\tau \sim \pi_\theta}[G(\tau)] \\
   &= \nabla_\theta \sum_\tau  \Pr(\tau|\theta) \, G(\tau) \\
   &= \sum_\tau \nabla_\theta \Pr(\tau|\theta) \, G(\tau) \\
   &= \sum_\tau \Pr(\tau|\theta)\,\nabla_\theta \log \Pr(\tau|\theta) \, G(\tau) \\
   &= \mathbb{E}_{\tau \sim \pi_\theta}\left[ \nabla_\theta \log \Pr(\tau|\theta) \, G(\tau) \right]
   \end{align*}
   $$
   where we made use of the log-derivative trick: $\nabla_\theta p(x) = p(x) \nabla_\theta \log p(x)$.
  
 
   The dynamics $\mu$ do not depend on $\theta$, so:
   $$
   \begin{align*}
   \log \Pr(\tau|\theta) &= \log \left( \prod_{t=0}^{T-1} \pi_\theta(a_t|s_t)\,\mu(s_{t+1}|s_t, a_t) \right) \\
   &= \sum_{t=0}^{T-1} \log \pi_\theta(a_t|s_t) + \sum_{t=0}^{T-1} \log \mu(s_{t+1}|s_t, a_t) \\
   &= \sum_{t=0}^{T-1} \log \pi_\theta(a_t|s_t) + \text{const.}
   \end{align*}
   $$
   where the const. term is independent of $\theta$, so when we take the gradient, it vanishes.

   Thus:
   $$
   \nabla_\theta \log \Pr(\tau|\theta) = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)
   $$

Plugging back into the gradient:
$$
\nabla_\theta J(\theta) =
\mathbb{E}_{\tau \sim \pi_\theta} \left[
  \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)\, G(\tau)
\right]
$$

This is the **Vanilla Policy Gradient estimator**, also called **REINFORCE**. 
Each $\log \pi_\theta(a_t|s_t)$ is multiplied by the **full return** $G(\tau)$. 
However, the action $a_t$ cannot influence rewards before time $t$, only those afterwards.
This means that all the rewards before timestep $t$ merely add noise, as no changes to the policy
can affect them.  To reduce variance, replace $G(\tau)$ with the return $G_t$ at timestep $t$, 
also called the **reward-to-go**:
$$
G_t = \sum_{i=t}^{T} \gamma^{i-t} r_{i}
$$

Thus, the lower-variance unbiased estimator is:
$$
\nabla_\theta J(\theta) =
\mathbb{E}_{\tau \sim \pi_\theta} \left[
  \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)\, G_t
\right]
$$

</details>

There are many other variants of the policy gradient estimator, as described in [Schulman, 2018](https://arxiv.org/abs/1506.02438).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/img/policy_grad.png" width="800">

## Implementation

We make use of the same CartPole environment as before, but now we have a vectorized version that is entirely defined in terms of tensor operations (see `chapter2_rl/exercises/gpu_env.py`). This environment is identical to the one used for DQN, but it now runs entirely on the GPU. This means
* we don't need to constantly convert between numpy and torch tensors
* we can run large numbers of environments in parallel (~thousands of environments for ~millions of environmental steps per second)
* we avoid copying data back and forth between the CPU and GPU, which can be a significant bottleneck

## Policy Network

Here, the policy is learned directly as a neural network, rather than learning a Q-value table approximator. We'll use the same architecture as the Q-network from DQN, so we've just included that here for you.

In [ ]:
class PolicyNetwork(nn.Module):
    """
    For consistency with your tests, please wrap your modules in a `nn.Sequential` called `layers`.
    """

    layers: nn.Sequential

    def __init__(self, obs_shape: tuple[int], num_actions: int, hidden_sizes: list[int] = [120, 84]):
        super().__init__()
        # assert len(obs_shape) == 1, f"Expecting a single vector of observations, got {obs_shape}"
        assert len(hidden_sizes) == 2, f"Expecting 2 hidden layers, got {len(hidden_sizes)}"
        self.layers = nn.Sequential(
            nn.Linear(obs_shape[-1], hidden_sizes[0]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Linear(hidden_sizes[1], num_actions),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)


net = PolicyNetwork(obs_shape=(4,), num_actions=2)
summary(net)

## Rollout Buffer

The way that our implementation of VPG will work is simple: we perform a rollout acrosss `num_envs` many environments in parallel, and store the trajectories for each. We then learn from that set of rollouts, and then discard it afterwards. One rollout, one learning step. This means we are always learning **on-policy**: we only every learn from data that the current model actually generated. We will use a rollout buffer to store the trajectories.

### Exercise - implement `Rollout Buffer`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 20 minutes on this exercise.
> ```

The `Rollout` class will store a set of `num_envs` many trajectories. WE do not shuffle up anything, or break up a episode into little experiences as we did for DQN. The smallest datapoint is one full trajectory:

$$\tau = s_0 \; a_0 \; r_0 \; s_1 \; a_1 \; r_1 \ldots s_T \; a_T \; r_T$$

The following methods need to be completed:

* `__init__` - initializes the rollout buffer, to store `obs`, `actions`, `logprobs`, `rewards`, `dones`, `entropy`, `infos`, `timestep`.
* `add_step` - adds information gathered from timestep $t$ to the rollout buffer
* `get_batches` - returns a list of `RolloutTensors` objects, each containing `batch_size` many trajectories.

<details>
<summary>Hint</summary>

Use `t.split` to write `get_batches`.
</details>

In [ ]:
RolloutTensors = namedtuple("RolloutTensors", ["obs", "actions", "logprobs", "rewards", "dones"])


class Rollout:
    obs: Float[Tensor, " num_envs max_size *obs_shape"]
    actions: Int[Tensor, " num_envs max_size *action_shape"]
    logprobs: Float[Tensor, " num_envs max_size"]
    rewards: Float[Tensor, " num_envs max_size"]
    dones: Bool[Tensor, " num_envs max_size"]
    infos: dict[str, Any]
    timestep: int

    def __init__(
        self, num_envs: int, max_steps: int, obs_shape: tuple[int], action_shape: tuple[int], device: t.device
    ):
        """
        Args:
            num_envs: number of environments to rollout
            max_steps: maximum number of steps to rollout per environment
            obs_shape: shape of the observation
            action_shape: shape of the action
            device: device to use
        """

        self.MAX_SIZE = max_steps
        # self.max_rollout_steps = args.max_rollout_steps
        # self.min_rollout_steps = args.min_rollout_steps

        self.obs = t.empty([num_envs, self.MAX_SIZE, *obs_shape], dtype=t.float32, device=device)
        self.actions = t.empty([num_envs, self.MAX_SIZE, *action_shape], dtype=t.int64, device=device)
        self.logprobs = t.empty([num_envs, self.MAX_SIZE], dtype=t.float32, device=device)
        self.rewards = t.empty([num_envs, self.MAX_SIZE], dtype=t.float32, device=device)
        self.dones = t.empty([num_envs, self.MAX_SIZE], dtype=t.bool, device=device)
        self.infos = {}
        self.timestep = 0

        self.tensors = RolloutTensors(self.obs, self.actions, self.logprobs, self.rewards, self.dones)

    def add_step(
        self,
        obs: Float[Tensor, " num_envs *obs_shape"],
        actions: Int[Tensor, " num_envs *action_shape"],
        logprobs: Float[Tensor, " num_envs"],
        rewards: Float[Tensor, " num_envs"],
        dones: Bool[Tensor, " num_envs"],
        infos: dict[str, Any],
    ):
        """
        Adds information to the replay buffer for the current self.timestep
        Don't forget to increment self.timestep afterwards!
        """

        if self.timestep >= self.MAX_SIZE:
            raise ValueError("Rollout is full, cannot add more steps")

        self.obs[:, self.timestep] = obs
        self.actions[:, self.timestep] = actions
        self.logprobs[:, self.timestep] = logprobs
        self.rewards[:, self.timestep] = rewards
        self.dones[:, self.timestep] = dones
        self.infos[self.timestep] = infos
        self.timestep += 1

    def reset(self):
        self.timestep = 0

    def get(self) -> tuple[Tensor, ...]:
        assert self.timestep == self.MAX_SIZE, "Rollout is not full"
        return self.tensors

    def get_batches(self, batch_size: int) -> list[RolloutTensors]:
        """
        Splits the rollout buffer into batches of size `batch_size`, and returns a list of
        `RolloutTensors` objects, each containing `batch_size` many trajectories.
        """

        obs = t.split(self.obs, batch_size, dim=0)
        acts = t.split(self.actions, batch_size, dim=0)
        logprobs = t.split(self.logprobs, batch_size, dim=0)
        rewards = t.split(self.rewards, batch_size, dim=0)
        dones = t.split(self.dones, batch_size, dim=0)

        batches = [RolloutTensors(*tensors) for tensors in zip(obs, acts, logprobs, rewards, dones)]

        return batches

## VPG Args

We've provided a dataclass for the training arguments, and will explain as needed later on.

In [ ]:
@dataclass
class VPGArgs:
    # Basic / global
    seed: int = 1
    env_id: str = "CartPole-gpu"

    # Wandb / logging
    use_wandb: bool = False
    wandb_project_name: str = "VPGCartPole"
    wandb_entity: str | None = None
    video_log_freq: int | None = 50

    # Duration of different phases / buffer memory settings
    total_timesteps: int = 500_000
    # max_rollout_steps: int = 500
    # min_rollout_steps: int = 64
    num_envs: int = 4

    num_steps_per_rollout: int = 128

    lr: float = 2.5e-4
    gamma: float = 1
    frac_dead_rollout: float = 1
    ent_coef: float = 0.01
    max_grad_norm: float = 0.5

    rollout_use_count: int = 4
    num_minibatches: int = 4
    clip_coef: float = 0.2
    compile: bool = False
    device: str = "cpu"
    normalize_returns: bool = True
    show_probs: bool = False
    num_batches_per_rollout: int = 1
    # LR decay settings
    use_lr_decay: bool = False
    lr_end: Optional[float] = None
    lr_frac: Optional[float] = None
    use_iw: bool = False

    def __post_init__(self):
        self.batch_size = self.num_envs // self.num_batches_per_rollout
        self.device = t.device(self.device)

        if self.use_lr_decay:
            assert self.lr_end is not None, "lr_end must be set if use_lr_decay is True"
            assert self.lr_frac is not None, "lr_frac must be set if use_lr_decay is True"

        self.env_steps_per_update = self.num_steps_per_rollout * self.num_envs // self.num_batches_per_rollout

        if not self.use_iw:
            assert self.rollout_use_count == 1, "rollout_use_count must be 1 if use_iw is False"
            assert self.num_batches_per_rollout == 1, "num_batches_per_rollout must be 1 if use_iw is False"

## VPG Agent

The following class will be out agent, that will generate rollouts via interaction between the agent and environment, as well as generate actions my sampling them from the policy network. Recall that the policy network now maps observations to logits for each action, so we can sample actions from the distribution.

### Exercise - implement `VPGAgent`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Implement the functions:
* `gen_rollout` - this function compute the episode rollout, by interacting with the environment for `args.num_steps_per_rollout` steps. If an episode terminates, we reset the environment and continue. We will track the length of the episode in the `lifespan` variable, which indicates how long each episode runs before termination. FOr the cartpole environment, this will allow us to track performance (the longer the cart lives, the better it does.)

* `get_actions` - this function takes in an observation, and returns the actions, logprobs, and entropy for that observation. You can use `t.distributions.Categorical(logits=logits)` to construct a distribution, from which you can get the actions, logprobs, and entropy. [See the docs](https://docs.pytorch.org/docs/stable/distributions.html#torch.distributions.categorical.Categorical) for details.

In [ ]:
class VPGAgent:
    """Base Agent class handling the interaction with the environment."""

    dead: Bool[Tensor, " num_envs"]
    lifespan: Int[Tensor, " num_envs"]

    def __init__(
        self,
        envs: gym.Env,
        policy_network: PolicyNetwork,
        args: VPGArgs,
        rng: Optional[np.random.Generator] = None,
    ):
        self.envs = envs
        self.policy_network = policy_network
        self.rng = rng
        self.args = args
        self.obs_shape = envs.observation_space.shape
        self.action_shape = envs.action_space.shape

    @t.no_grad()
    def gen_rollout(self, rollout: Rollout) -> tuple[Rollout, dict[str, Any]]:
        """
        Compute the full episode rollout for all environments in parallel, adding them to the rollout buffer.
        It then returns the rollout buffer, and a dictionary of info contining the lifespan.

        Returns `infos` (list of dictionaries containing info we will log).
        """
        obs, _ = self.envs.reset()  # Need a starting observation
        device = self.args.device

        dead = t.zeros(self.args.num_envs, dtype=t.bool, device=device)
        lifespan = t.zeros(self.args.num_envs, dtype=t.int32, device=device)
        rollout.reset()

        for timestep in range(self.args.num_steps_per_rollout):
            actions, logprobs, entropy = self.get_actions(obs)
            new_obs, rewards, terminates, _, info = self.envs.step(actions)
            done = terminates
            rollout.add_step(obs, actions, logprobs, rewards, done, info)
            obs = new_obs
            dead = dead | done
            lifespan += ~dead

        info = {"lifespan": lifespan}

        return rollout, info

    def get_actions(
        self, obs: Float[Tensor, " num_envs *obs_shape"]
    ) -> tuple[Int[Tensor, " num_envs *action_shape"], Float[Tensor, " num_envs"], Float[Tensor, " num_envs"]]:
        """
        Computes the agents turn: given an observation for eahc environment,
        sample the action the agent takes, along with the log_probs of that action,
        and the entropy of the action distribution.
        """
        logits = self.policy_network(obs)
        dist = t.distributions.Categorical(logits=logits)
        actions = dist.sample()
        entropy = dist.entropy()
        logprobs = dist.log_prob(actions)
        return actions, logprobs, entropy

## Returns

To compute the REINFORCE loss, we need to compute the return for each step in the trajectory. This gets a little messy as trajectories may be of different lengths, so an episode may have terminated part way through the rollout. You'll need to walk backward through the trajectory, and compute the return for each step.

### Exercise - implement `compute_returns`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Compute the returns for each trajectory. Easiest to write as a simple reverse for-loop for now, though if you wish later on you can try a vectorized solution.

In [ ]:
def compute_returns(
    rewards: Float[Tensor, " num_envs num_steps"], done: Bool[Tensor, " num_envs num_steps"], gamma: float = 0.9
):
    """
    ARGS:
        rewards: The rewards for each trajectory
        done: A boolean tensor indicating if an episode finished on the current timestep
        gamma: The discount factor

    Returns:
        The returns G_t for each trajectory.

        For example:
        - If Rewards = [0, 0, 1, 0, 1]
        - And Done   = [0, 0, 1, 0, 1]
        - Then Returns = [g**2 + g + 1, g + 1, 1, g, 1]
    """
    num_envs, num_steps = rewards.shape

    returns = t.zeros_like(rewards)


    G = t.zeros_like(rewards[:, 0])  # (num_envs)
    for i in reversed(range(num_steps)):
        G = rewards[:, i] + gamma * G * (~done[:, i])
        returns[:, i] = G
    return returns


tests.test_compute_returns(compute_returns)

### Exercise - implement `compute_logprobs_and_entropy`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Computes the logprobs of actions taken, and the entropy of the action distribution on each timestep. Needed for the loss function.

In [ ]:
def compute_logprobs_and_entropy(
    tau: RolloutTensors, pi: PolicyNetwork
) -> tuple[Float[Tensor, " num_envs num_steps"], Float[Tensor, " num_envs num_steps"]]:
    """
    Computes the logprobs and entropy of the action distribution on each timestep.
    """
    logits = pi(tau.obs)
    log_probs = F.log_softmax(logits, dim=-1)
    log_probs_taken = eindex(log_probs, tau.actions, "env time [env time] -> env time")
    probs_taken = log_probs_taken.exp()
    entropy = -(probs_taken * log_probs_taken).sum(dim=-1)
    return log_probs_taken, entropy

## Building up to the loss function

We need to compute the probability ratio $\pi(a_t | s_t) / \pi_{old}(a_t | s_t)$ for each timestep taken in the rollout. This is used to compute the importance weights $\text{iw}_t$, which allows us to learn off-policy. If `args.clip_coef` is not none, we also clamp the importance weights between `1 - args.clip_coef` and `1 + args.clip_coef`.

### Exercise - implement `compute_importance_weights`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Keep the result numerically stable by exponentiating the difference between the logprobs.
Gradietns should **NOT** flow through the importance weights. Make sure to use `.detach()` to prevent this.

In [ ]:
def compute_importance_weights(logprobs_taken, tau: RolloutTensors, clip_coef: Optional[float]) -> t.Tensor:
    iw = t.exp(logprobs_taken - tau.logprobs).detach()  # Detach to prevent gradient flow
    if clip_coef is not None:
        iw = t.clamp(iw, 1 - clip_coef, 1 + clip_coef)
    return iw

### Exercise - implement `normalize_returns`

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 5 minutes on this exercise.
> ```

Normalize the returns by ensuring zero mean, unit variance **across all trajectories and timesteps**. Don't overthink this one, should be a one-liner. Don't worry about having the episode inside or outside the square root in the denominator. Doesn't really matter.

In [ ]:
def normalize_returns(returns: Float[Tensor, " num_envs num_steps"]) -> Float[Tensor, " num_envs num_steps"]:
    """
    Normalizes the returns by ensuring zero mean, unit variance across all trajectories and timesteps.
    """
    return (returns - returns.mean()) / (returns.std() + 1e-8)

### Exercise - implement `compute_reinforce_loss`

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 5 minutes on this exercise.
> ```

This should be easy with everything else you've got.
The loss on timestep $t$ is 
$$
\rho_t \log \pi(a_t | s_t) \big( G_t - b(s_t) \big)
$$
where $G_t$ is the return, $\rho_t$ is the importance weight, and $\log \pi(a_t | s_t)$ are the logprobs, each for timestep $t$, and $b(s_t)$ is some baseline function thet depends on the state $s_t$. For now, we will just use the average return for each trajectory as the baseline. PPO uses a more advanced baseline function claled a critic, that we will see tomorrow.

The total loss is the mean of the losses over all timesteps, over all trajectories.

In [ ]:
def compute_reinforce_loss(
    returns: Float[Tensor, " num_envs num_steps"],
    logprobs_taken: Float[Tensor, " num_envs num_steps"],
    iw: Float[Tensor, " num_envs num_steps"],
) -> Float[Tensor, ""]:
    target = returns - returns.mean()
    return (iw * logprobs_taken * target.detach()).mean()

## Trainer

This is the function that will handle the full training loop. We've provided you with the template of a training loop which should be very similar to yesterday's.

### Exercise - implement `VPGTrainer`

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 45 minutes on this exercise.
> ```

You should fill in the following methods. Ignore logging, can just copy from the solution later.

* `compute_loss` - this method should compute the loss for the VPG objective function.

The training loop is rather standard once everything else is done: we do a rollout, we cut the result into batches, compute the loss, and update the weights from each batch, so we've just included that gor

In [ ]:
class VPGTrainer:
    def __init__(self, args: VPGArgs):
        set_global_seeds(args.seed)
        self.args = args

        device = args.device

        self.rng = t.Generator(device=device).manual_seed(args.seed)
        self.run_name = f"{args.env_id}__{args.wandb_project_name}__seed{args.seed}__{time.strftime('%Y%m%d-%H%M%S')}"

        if args.env_id == "CartPole-gpu":
            self.envs = CartPole(args.num_envs, device=device)
        elif args.env_id == "Probe4-v0":
            self.envs = Probe4(args.num_envs)
        elif args.env_id == "Probe5-v0":
            self.envs = Probe5(args.num_envs)
        else:
            raise ValueError(f"Environment {args.env_id} not supported")

        # Define some basic variables from our environment (note, we assume a single discrete action space)
        self.num_envs = args.num_envs
        self.action_shape = self.envs.action_space.shape
        self.num_actions = self.envs.action_space.n
        self.obs_shape = self.envs.observation_space.shape

        # Create our networks & optimizer
        self.policy_network = PolicyNetwork(self.obs_shape, self.num_actions).to(device)

        # Compile the policy network for faster inference
        if self.args.compile:
            self.policy_network = t.compile(self.policy_network)

        self.optimizer = t.optim.Adam(self.policy_network.parameters(), lr=args.lr, eps=1e-5, maximize=True)
        self.optimizer.zero_grad()

        # Create our agent
        self.agent = VPGAgent(envs=self.envs, policy_network=self.policy_network, args=self.args, rng=self.rng)

    def compute_loss(self, tau: RolloutTensors) -> tuple[t.Tensor, dict[str, Any]]:
        returns = compute_returns(tau.rewards, tau.dones, self.args.gamma)  # (num_envs, timestep)

        if self.args.normalize_returns:
            returns = normalize_returns(returns)

        logprobs_taken, entropy = compute_logprobs_and_entropy(tau, self.policy_network)

        iw = compute_importance_weights(logprobs_taken, tau, self.args.clip_coef)
        r_joy = compute_reinforce_loss(returns, logprobs_taken, iw)
        avg_entropy = entropy.mean()

        joy = r_joy + self.args.ent_coef * avg_entropy


        info = {
            "entropy": avg_entropy.item(),
            "r_joy": r_joy.item(),
            "iw": iw.mean().item() if self.args.use_iw else None,
        }

        return joy, info

    def update_learning_rate(self, time_steps, args):
        if args.use_lr_decay and args.lr_frac > 0:
            progress = min(1.0, max(time_steps / args.total_timesteps, 0) / args.lr_frac)
            return (progress * args.lr_end) + ((1 - progress) * args.lr)
        return args.lr

    def train(self) -> None:
        """
        Trains the agent by generating rollouts and updating the policy.
        The progress bar tracks total environment steps.
        """
        # --- Setup ---
        rollout = Rollout(
            num_envs=self.num_envs,
            max_steps=self.args.num_steps_per_rollout,
            obs_shape=self.obs_shape,
            action_shape=self.action_shape,
            device=self.args.device,
        )

        # Calculate the number of environment steps collected per rollout generation

        # Calculate the total number of updates (rollouts) to perform
        # Use integer division to ensure we don't exceed total_timesteps

        env_steps_per_train_step = (
            self.args.num_steps_per_rollout * self.args.num_envs // (self.args.num_batches_per_rollout)
        )

        num_updates = self.args.total_timesteps // env_steps_per_train_step
        train_steps = 0  # Counter for gradient updates

        # --- Training Loop ---
        # The progress bar is managed manually with a `with` statement.
        # `total` is set to the total environment steps we want to run.
        # The loop iterates `num_updates` times, not `total_timesteps` times.
        with tqdm(
            total=self.args.total_timesteps,
            unit=" env steps",
            unit_scale=True,
            desc="Training",
            miniters=1,
            mininterval=0.02,
        ) as pbar:
            env_steps_consumed = 0

            for update_num in range(num_updates):
                # 1. Generate a new rollout from the environment

                rollout, agent_info = self.agent.gen_rollout(rollout)

                # 2. Split the rollout into batches along the num_envs dimension

                rollout_batches = rollout.get_batches(self.args.batch_size)

                # 3. Logging and Progress Bar Update
                # This part is outside the inner loop to only log once per rollout
                avg_lifespan = agent_info["lifespan"].float().mean().item()
                std_lifespan = agent_info["lifespan"].float().std().item()
                max_lifespan = agent_info["lifespan"].max().item()

                if (avg_lifespan + 0.5) > self.args.num_steps_per_rollout and std_lifespan < 0.01:
                    print("Agent has learned to play optimally!")
                    break

                # 4. For each batch, perform multiple gradient updates
                for batch in rollout_batches:
                    for i in range(self.args.rollout_use_count):
                        loss, reinforce_info = self.compute_loss(batch)

                        info = {**agent_info, **reinforce_info}

                        loss.backward()
                        if self.args.max_grad_norm is not None:
                            t.nn.utils.clip_grad_norm_(
                                self.policy_network.parameters(), max_norm=self.args.max_grad_norm
                            )

                        grad_norm = t.nn.utils.clip_grad_norm_(self.policy_network.parameters(), max_norm=float("inf"))

                        self.optimizer.step()
                        self.optimizer.zero_grad()
                        train_steps += 1

                        new_lr = self.update_learning_rate(pbar.n, self.args)

                        for pg in self.optimizer.param_groups:
                            pg["lr"] = new_lr

                        # Create info string to display in the progress bar
                        current_lr = self.optimizer.param_groups[0]["lr"]
                        info_dict = {
                            "joy": f"{info['r_joy']:.4f}",
                            "traj_len": f"{avg_lifespan:.2f} ± {std_lifespan:.2f} (max: {max_lifespan:.2f})",
                            "H": f"{info['entropy']:.4f}",
                            "iw": f"{info['iw']:.4f}" if self.args.use_iw else None,
                            "∇": f"{grad_norm:.4f}",
                            "lr": f"{current_lr:.2e}",
                        }

                        pbar.set_postfix(info_dict)
                        pbar.update(env_steps_per_train_step)

        # --- Cleanup ---
        self.envs.close()
        if self.args.use_wandb:
            wandb.finish()

## Probes

As yesterday, we will be using probes to test our model. They've been implemented for you.

In [ ]:
def test_probe(probe_idx: int):
    """
    Tests a probe environment by training a network on it & verifying that the value functions are
    in the expected range.
    """
    # Train our network
    args = VPGArgs(
        env_id=f"Probe{probe_idx}-v0",
        wandb_project_name=f"test-probe-{probe_idx}",
        total_timesteps=[7500, 7500, 12500, 20000, 20000][probe_idx - 1],
        lr=5e-3,
        num_envs=4,
        video_log_freq=None,
        use_wandb=False,
        device="cpu",
        ent_coef=0.0,
        clip_coef=None,
        normalize_returns=False,
        rollout_use_count=1,
        show_probs=True,
    )
    trainer = VPGTrainer(args)
    trainer.train()
    agent = trainer.agent

    # Get the correct set of observations, and corresponding values we expect
    obs_for_probes = [[[0.0]], [[-1.0], [+1.0]], [[0.0], [1.0]], [[0.0]], [[0.0], [1.0]]]
    expected_value_for_probes = [
        [[1.0]],
        [[-1.0], [+1.0]],
        [[args.gamma], [1.0]],
        [[1.0]],
        [[1.0], [1.0]],
    ]
    expected_probs_for_probes = [None, None, None, [[0.0, 1.0]], [[1.0, 0.0], [0.0, 1.0]]]
    tolerances = [1e-3, 1e-3, 1e-3, 2e-3, 2e-3]
    obs = t.tensor(obs_for_probes[probe_idx - 1]).to(args.device)

    # Calculate the actual value & probs, and verify them
    with t.inference_mode():
        probs = agent.policy_network(obs).softmax(-1)
    expected_probs = expected_probs_for_probes[probe_idx - 1]
    if expected_probs is not None:
        print(f"Probs: {probs}")
        print(f"Expected probs: {t.tensor(expected_probs).to(args.device)}")
        t.testing.assert_close(probs, t.tensor(expected_probs).to(args.device), atol=tolerances[probe_idx - 1], rtol=0)
    print(f"Probe {probe_idx} tests passed!\n")


gym.envs.registration.register(id="Probe4-v0", entry_point=Probe4)
gym.envs.registration.register(id="Probe5-v0", entry_point=Probe5)

for probe_idx in [4, 5]:
    test_probe(probe_idx)

## Training Run

Vanilla Policy Gradient can often be a bit finicky and unstable to train (which is why in practice we use PPO instead). None-the-less, I've tried to find a good set of hyperparameters that work reasoanbly okay, and a set that (if you're lucky), trains to optimality in ~15 seconds on CartPole!

In [ ]:
args = VPGArgs(use_wandb=False,
            num_envs=4,
            num_batches_per_rollout=1,
            total_timesteps=500_000,
            num_steps_per_rollout=500,
            rollout_use_count=4,  # this seems to matter a lot
            ent_coef=0.3, #works with zero
            clip_coef=0.2, #can sometimes work with no clipping, but it helps
            max_grad_norm=0.5,
            normalize_returns=True,
            use_iw = True,
            lr = 1e-4,
            gamma=0.99,
            device="cpu") #may run faster on cpu due to few envs/small batchsize
trainer = VPGTrainer(args)
trainer.train()
generate_and_plot_trajectory(trainer, args, mode = "pg")

In [ ]:
# There's a somewhat critical region where the cartpole really picks up,
# and we need the LR to decay rather fast before the gradients explode
# No guarantees that this will work for other environments, but it's a good starting point
# sub 15 seconds to optimal on A4000!!
# might need to rerun a few times to get a lucky initialization, it's rather sensitive!
device = t.device("cuda")

args_fast = VPGArgs(
    use_wandb=False,
    num_envs=256,
    num_batches_per_rollout=4,
    total_timesteps=4_000_000,
    num_steps_per_rollout=500,
    rollout_use_count=1,  # this seems to matter a lot
    ent_coef=0.5,  # works with zero
    clip_coef=0.1,  # can sometimes work with no clipping, but it helps
    max_grad_norm=1,
    normalize_returns=True,
    lr=1e-2,  # risky!
    use_lr_decay=True,
    use_iw=True,  # dont' need it if we only use each rollout once in one
    lr_end=1e-3,
    lr_frac=0.6,
    compile=False,
    gamma=0.99,
    seed=1337,
    device=device,
)

trainer = VPGTrainer(args_fast)
trainer.train()
generate_and_plot_trajectory(trainer, args, mode="pg")